## Setup

In [1]:
from os.path  import join
import random
import itertools
from itertools import product
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
MenTrain = pd.read_csv('MenTrain.csv', index_col=0)
MenTest = pd.read_csv('MenTest.csv', index_col=0)
MenTest['location'] = 'location' # to remove NaNs

WomenTrain = pd.read_csv('WomenTrain.csv', index_col=0)
WomenTest = pd.read_csv('WomenTest.csv', index_col=0)
WomenTest['location'] = 'location' # to remove NaNs

In [3]:
data_men = pd.concat([MenTrain.dropna(), MenTest.dropna()])
data_women = pd.concat([WomenTrain.dropna(), WomenTest.dropna()])

## Function to test models on

In [4]:
def test_models(data_men, data_women, start_year, end_year, models):
    train_years = [year for year in range(start_year, end_year)]
    train_years.remove(2019)
    train_years.remove(2020)
    test_years = [year+1 for year in train_years]

    # Store scores for best model (based on mean score)
    best_mean_brier_score_based_on_mean = 1.0
    best_final_season_brier_score_based_on_mean = 1.0
    best_parameters_based_on_mean = None
    # Store scores for best model (based on final season)
    best_mean_brier_score_based_on_final_season = 1.0
    best_final_season_brier_score_based_on_final_season = 1.0
    best_parameters_based_on_final_season = None
    for model in models:
        brier_scores = []

        print('----------------------------------------------------------')
        print('Model parameters:', model.get_params())
        print()
        for train_year, test_year in zip(train_years, test_years):

            y_test_all = []
            y_pred_all = []
            
            # For men data __________________________________
            data = data_men
            # Prepare data ---------------------------------
            # Get train test subsest
            data_train = data[data['Season']==train_year]
            data_test = data[data['Season']==test_year]
            # Get y train and test label
            y_train = ((data_train['T1_Score'] > data_train['T2_Score']).values).astype(int)
            y_test = ((data_test['T1_Score'] > data_test['T2_Score']).values).astype(int)
            # Get X train and test data
            X_train = data_train.iloc[:, 6:].values
            X_test = data_test.iloc[:, 6:].values
            # Evaluate model -------------------------------
            # Fit model
            model.fit(X_train, y_train)
            # Predict
            if hasattr(model, "predict_proba"):
                y_pred = model.predict_proba(X_test)[:, 1]
            elif hasattr(model, "decision_function"):
                decision = model.decision_function(X_test)
                y_pred = (decision - decision.min()) / (decision.max() - decision.min())
            else:
                y_pred = model.predict(X_test)
            y_pred[y_pred > 0.6] = 1
            y_pred[y_pred < 0.2] = 0
            y_test_all.extend(y_test)
            y_pred_all.extend(y_pred)
            # Print results --------------------------------
            print('For men train year:', train_year, ', test year:', test_year, 'brier score:', np.mean((y_test - y_pred)**2))

            # For women data _______________________________
            data = data_women
            # Prepare data ---------------------------------
            # Get train test subsest
            data_train = data[data['Season']==train_year]
            data_test = data[data['Season']==test_year]
            # Get y train and test label
            y_train = ((data_train['T1_Score'] > data_train['T2_Score']).values).astype(int)
            y_test = ((data_test['T1_Score'] > data_test['T2_Score']).values).astype(int)
            # Get X train and test data
            X_train = data_train.iloc[:, 6:].values
            X_test = data_test.iloc[:, 6:].values
            # Evaluate model -------------------------------
            # Fit model
            model.fit(X_train, y_train)
            # Predict
            if hasattr(model, "predict_proba"):
                y_pred = model.predict_proba(X_test)[:, 1]
            elif hasattr(model, "decision_function"):
                decision = model.decision_function(X_test)
                y_pred = (decision - decision.min()) / (decision.max() - decision.min())
            else:
                y_pred = model.predict(X_test)
            y_pred[y_pred > 0.7] = 1
            y_pred[y_pred < 0.2] = 0
            y_test_all.extend(y_test)
            y_pred_all.extend(y_pred)
            # Print results --------------------------------
            print('For women train year:', train_year, ', test year:', test_year, 'brier score:', np.mean((y_test - y_pred)**2))

            # Results for men and women
            y_test_all = np.array(y_test_all)
            y_pred_all = np.array(y_pred_all)
            brier_scores.append(np.mean((y_test_all - y_pred_all)**2))
            print('Results for both men and women:', brier_scores[-1]) 

        # Get aggregated results ---------------------------
        if np.mean(brier_scores) < best_mean_brier_score_based_on_mean:
            best_mean_brier_score_based_on_mean = np.mean(brier_scores)
            best_final_season_brier_score_based_on_mean = brier_scores[-1]
            best_parameters_based_on_mean = model.get_params()
        if brier_scores[-1] < best_final_season_brier_score_based_on_final_season:
            best_mean_brier_score_based_final_season = np.mean(brier_scores)
            best_final_season_brier_score_based_on_final_season = brier_scores[-1]
            best_parameters_based_final_season = model.get_params()
        print()
        print('The mean score across seasons was', np.mean(brier_scores), 'with std of', np.std(brier_scores))

    # Print final report ----------------------------------
    print()
    print('############################ FINAL REPORT ############################')
    print('The best model based on mean brier score was:', best_parameters_based_on_mean)
    print('The mean brier score for this model was', best_mean_brier_score_based_on_mean,
          'and the score for', end_year, 'year was', best_final_season_brier_score_based_on_mean)
    print()
    print('The best model based on', end_year, "year's brier score was:", best_parameters_based_final_season)
    print('The mean brier score for this model was', best_mean_brier_score_based_final_season,
          'and the score for', end_year, 'year was', best_final_season_brier_score_based_on_final_season)

## Logistic regression

In [5]:
param_grid = {
    'C': [0.01, 0.1, 1.0],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'class_weight': [None, 'balanced'],
    'max_iter': [300, 1000],
    'random_state': [42],
}
def is_valid(params):
    if params['penalty'] == 'l2':
        return params['solver'] in ['lbfgs', 'liblinear']
    return False
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in product(*values)]
param_list = [p for p in param_combinations if is_valid(p)]

models = [LogisticRegression(**params) for params in param_list]
test_models(data_men=data_men, data_women=data_women, start_year=2010, end_year=2025, models=models)

----------------------------------------------------------
Model parameters: {'C': 0.01, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 300, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

For men train year: 2010 , test year: 2011 brier score: 0.25939918098845677
For women train year: 2010 , test year: 2011 brier score: 0.12723808964867198
Results for both men and women: 0.19535188287763797
For men train year: 2011 , test year: 2012 brier score: 0.22851865775645938
For women train year: 2011 , test year: 2012 brier score: 0.1449241252141456
Results for both men and women: 0.1880074612167227
For men train year: 2012 , test year: 2013 brier score: 0.20185823830271954
For women train year: 2012 , test year: 2013 brier score: 0.20961071748161506
Results for both men and women: 0.20561520898172272
For men train year: 2013

## KNN

In [6]:
param_grid = {
    'n_neighbors': [3, 5, 7, 10],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree'],
    'p': [1, 2],
    'leaf_size': [20, 30, 40],
}
keys, values = zip(*param_grid.items())
param_list = [dict(zip(keys, v)) for v in product(*values)]

models = [KNeighborsClassifier(**params) for params in param_list]
test_models(data_men=data_men, data_women=data_women, start_year=2010, end_year=2025, models=models)

----------------------------------------------------------
Model parameters: {'algorithm': 'auto', 'leaf_size': 20, 'metric': 'minkowski', 'metric_params': None, 'n_jobs': None, 'n_neighbors': 3, 'p': 1, 'weights': 'uniform'}

For men train year: 2010 , test year: 2011 brier score: 0.25621890547263676
For women train year: 2010 , test year: 2011 brier score: 0.15432098765432098
Results for both men and women: 0.20683760683760688
For men train year: 2011 , test year: 2012 brier score: 0.2437810945273632
For women train year: 2011 , test year: 2012 brier score: 0.1384479717813051
Results for both men and women: 0.19273504273504274
For men train year: 2012 , test year: 2013 brier score: 0.23963515754560533
For women train year: 2012 , test year: 2013 brier score: 0.17283950617283947
Results for both men and women: 0.20726495726495728
For men train year: 2013 , test year: 2014 brier score: 0.26616915422885573
For women train year: 2013 , test year: 2014 brier score: 0.1384479717813051
Resu

## SVM

In [5]:
param_grid = {
    'C': [0.1, 1.0],
    'kernel': ['rbf', 'poly'],
    'gamma': ['scale'],
    'degree': [2, 3],
    'class_weight': ['balanced'],
    'random_state': [42],
}
def is_valid(params):
    if params['kernel'] != 'poly' and 'degree' in params:
        return True
    return True
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in product(*values)]
param_list = [p for p in param_combinations if is_valid(p)]

models = [SVC(**params) for params in param_list]
test_models(data_men=data_men, data_women=data_women, start_year=2010, end_year=2025, models=models)

----------------------------------------------------------
Model parameters: {'C': 0.1, 'break_ties': False, 'cache_size': 200, 'class_weight': 'balanced', 'coef0': 0.0, 'decision_function_shape': 'ovr', 'degree': 2, 'gamma': 'scale', 'kernel': 'rbf', 'max_iter': -1, 'probability': False, 'random_state': 42, 'shrinking': True, 'tol': 0.001, 'verbose': False}

For men train year: 2010 , test year: 2011 brier score: 0.1926716179221841
For women train year: 2010 , test year: 2011 brier score: 0.15001779192147763
Results for both men and women: 0.172000917629534
For men train year: 2011 , test year: 2012 brier score: 0.16564745749989013
For women train year: 2011 , test year: 2012 brier score: 0.15134708340398006
Results for both men and women: 0.15871727620725679
For men train year: 2012 , test year: 2013 brier score: 0.19688243646073814
For women train year: 2012 , test year: 2013 brier score: 0.1630430238910161
Results for both men and women: 0.18048333652310364
For men train year: 2013